In [ ]:
!pip install -q numpy pandas scikit-learn librosa==0.10.2.post1
!pip install -q transformers tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import csv
import zipfile
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoFeatureExtractor
from tqdm.auto import tqdm
from google.colab import files

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE = 16000

In [ ]:
BASE_DIR = "/content/drive/MyDrive/MDD"

PRIVATE_ROOT = f"{BASE_DIR}/Data/Private/MDD-Challenge-2025-private-test/MDD-Challenge-2025-private-test"
PRIVATE_META_DIR = f"{PRIVATE_ROOT}/metadata"
PRIVATE_AUDIO_DIR = f"{PRIVATE_ROOT}/audio_data/private_test"

PRIVATE_CSV = f"{PRIVATE_META_DIR}/private_test_submission.csv"
CKPT_PATH = f"{BASE_DIR}/MDD_Work/checkpoints/best_mdd_vietnamese.pt"

OUT_DIR = f"{BASE_DIR}/Private_Result"
os.makedirs(OUT_DIR, exist_ok=True)

print("DEVICE:", DEVICE)
print("BASE_DIR:", os.path.exists(BASE_DIR), BASE_DIR)
print("PRIVATE_ROOT:", os.path.exists(PRIVATE_ROOT), PRIVATE_ROOT)
print("PRIVATE_META_DIR:", os.path.exists(PRIVATE_META_DIR), PRIVATE_META_DIR)
print("PRIVATE_AUDIO_DIR:", os.path.exists(PRIVATE_AUDIO_DIR), PRIVATE_AUDIO_DIR)
print("PRIVATE_CSV:", os.path.exists(PRIVATE_CSV), PRIVATE_CSV)
print("CKPT_PATH:", os.path.exists(CKPT_PATH), CKPT_PATH)

print("Metadata files:", os.listdir(PRIVATE_META_DIR))
print("Audio sample:", os.listdir(PRIVATE_AUDIO_DIR)[:10])

DEVICE: cpu
BASE_DIR: True /content/drive/MyDrive/MDD
PRIVATE_ROOT: True /content/drive/MyDrive/MDD/Data/Private/MDD-Challenge-2025-private-test/MDD-Challenge-2025-private-test
PRIVATE_META_DIR: True /content/drive/MyDrive/MDD/Data/Private/MDD-Challenge-2025-private-test/MDD-Challenge-2025-private-test/metadata
PRIVATE_AUDIO_DIR: True /content/drive/MyDrive/MDD/Data/Private/MDD-Challenge-2025-private-test/MDD-Challenge-2025-private-test/audio_data/private_test
PRIVATE_CSV: True /content/drive/MyDrive/MDD/Data/Private/MDD-Challenge-2025-private-test/MDD-Challenge-2025-private-test/metadata/private_test_submission.csv
CKPT_PATH: True /content/drive/MyDrive/MDD/MDD_Work/checkpoints/best_mdd_vietnamese.pt
Metadata files: ['private_test_submission_example.csv', 'README_submission.md', 'private_test_submission.csv', '.DS_Store']
Audio sample: ['thanh_98_10.wav', 'thanh_69_8.wav', 'thanh_50_6.wav', 'thanh_28_4.wav', 'thanh_93_10.wav', 'thanh_72_9.wav', 'thanh_6_1.wav', 'thay-giao_161873335770

In [ ]:
private_df = pd.read_csv(PRIVATE_CSV)

print(private_df.head())
print(private_df.shape)
print(private_df.columns.tolist())

assert "id" in private_df.columns
assert "path" in private_df.columns
assert "canonical" in private_df.columns

                        id                                               path  \
0  nang-giai_1618731750770  audio_data/private_test/nang-giai_161873175077...   
1  nang-giai_1618731781800  audio_data/private_test/nang-giai_161873178180...   
2  nang-giai_1618732605210  audio_data/private_test/nang-giai_161873260521...   
3  nang-giai_1618732684413  audio_data/private_test/nang-giai_161873268441...   
4  nang-giai_1618732695387  audio_data/private_test/nang-giai_161873269538...   

            canonical  
0  n a-0 ŋz z aː-3 iz  
1  n a-0 ŋz z aː-3 iz  
2  n a-0 ŋz z aː-3 iz  
3  n a-0 ŋz z aː-3 iz  
4  n a-0 ŋz z aː-3 iz  
(856, 3)
['id', 'path', 'canonical']


In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)

vocab = checkpoint["vocab"]
id2token = {v: k for k, v in vocab.items()}

vocab_size = len(vocab)
BLANK_ID = 0
UNK_ID = 1

PRETRAINED_MODEL = checkpoint["config"]["pretrained"]

print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Checkpoint score:", checkpoint.get("score"))
print("Pretrained model:", PRETRAINED_MODEL)
print("Vocab size:", vocab_size)
print(list(vocab.items())[:20])

Checkpoint epoch: 21
Checkpoint score: 0.4627452384257296
Pretrained model: nguyenvulebinh/wav2vec2-base-vietnamese-250h
Vocab size: 123
[('<blank>', 0), ('<unk>', 1), ('a-0', 2), ('a-1', 3), ('a-2', 4), ('a-3', 5), ('a-4', 6), ('a-5', 7), ('aː-0', 8), ('aː-1', 9), ('aː-2', 10), ('aː-3', 11), ('aː-4', 12), ('aː-5', 13), ('e-0', 14), ('e-1', 15), ('e-2', 16), ('e-3', 17), ('e-4', 18), ('e-5', 19)]


In [ ]:
def text_to_ids(text):
    ids = []
    for tok in str(text).split():
        if tok and tok != "$":
            ids.append(vocab.get(tok, UNK_ID))
    return ids


def greedy_decode(logits):
    pred_ids = torch.argmax(logits, dim=-1).detach().cpu().tolist()

    collapsed = []
    prev = None

    for idx in pred_ids:
        if idx != prev:
            collapsed.append(idx)
        prev = idx

    tokens = []
    for idx in collapsed:
        if idx == BLANK_ID:
            continue

        tok = id2token.get(idx, "")
        if tok not in ["<blank>", "<unk>", "$", ""]:
            tokens.append(tok)

    return " ".join(tokens)

In [ ]:
class AcousticLinguisticMDD(nn.Module):
    def __init__(self, pretrained_name, vocab_size):
        super().__init__()

        self.audio_encoder = AutoModel.from_pretrained(pretrained_name)
        hidden = self.audio_encoder.config.hidden_size

        if hasattr(self.audio_encoder, "feature_extractor"):
            for p in self.audio_encoder.feature_extractor.parameters():
                p.requires_grad = False

        self.phone_embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=BLANK_ID
        )

        self.phone_encoder = nn.LSTM(
            input_size=128,
            hidden_size=hidden // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=8,
            dropout=0.1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.2)

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, vocab_size)
        )

    def forward(self, input_values, canonical):
        audio_out = self.audio_encoder(input_values).last_hidden_state

        phone_emb = self.phone_embedding(canonical)
        phone_out, _ = self.phone_encoder(phone_emb)

        attn_out, _ = self.cross_attn(
            query=audio_out,
            key=phone_out,
            value=phone_out
        )

        fused = torch.cat([audio_out, attn_out], dim=-1)
        fused = self.dropout(fused)

        logits = self.classifier(fused)
        return logits


model = AcousticLinguisticMDD(PRETRAINED_MODEL, vocab_size).to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

print("Loaded model OK")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: nguyenvulebinh/wav2vec2-base-vietnamese-250h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model OK


In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(PRETRAINED_MODEL)


class PrivateMDDDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def resolve_wav_path(self, p):
        p = str(p)

        # TH1: path trong csv đã là audio_data/private_test/xxx.wav
        full1 = os.path.join(PRIVATE_ROOT, p)
        if os.path.exists(full1):
            return full1

        # TH2: path chỉ là private_test/xxx.wav
        full2 = os.path.join(PRIVATE_ROOT, "audio_data", p)
        if os.path.exists(full2):
            return full2

        # TH3: path chỉ là tên file xxx.wav
        full3 = os.path.join(PRIVATE_AUDIO_DIR, p)
        if os.path.exists(full3):
            return full3

        # TH4: path thiếu .wav
        full4 = os.path.join(PRIVATE_AUDIO_DIR, f"{p}.wav")
        if os.path.exists(full4):
            return full4

        raise FileNotFoundError(f"Không tìm thấy audio: {p}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        wav_path = self.resolve_wav_path(row["path"])
        wav, _ = librosa.load(wav_path, sr=SAMPLE_RATE)

        canonical_ids = text_to_ids(row["canonical"])

        return {
            "wav": wav,
            "canonical_ids": canonical_ids,
            "id": row["id"],
            "path": row["path"],
            "canonical": row["canonical"],
        }


def private_collate_fn(batch):
    wavs = [b["wav"] for b in batch]

    inputs = feature_extractor(
        wavs,
        sampling_rate=SAMPLE_RATE,
        padding=True,
        return_tensors="pt"
    )

    max_can_len = max(len(b["canonical_ids"]) for b in batch)

    canonical = torch.full(
        (len(batch), max_can_len),
        BLANK_ID,
        dtype=torch.long
    )

    for i, b in enumerate(batch):
        can = b["canonical_ids"]
        canonical[i, :len(can)] = torch.tensor(can, dtype=torch.long)

    return {
        "input_values": inputs.input_values,
        "canonical": canonical,
        "ids": [b["id"] for b in batch],
        "paths": [b["path"] for b in batch],
        "canonicals": [b["canonical"] for b in batch],
    }


private_ds = PrivateMDDDataset(private_df)

private_loader = DataLoader(
    private_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=private_collate_fn,
    num_workers=0
)

print("Private samples:", len(private_ds))

for i in range(min(5, len(private_ds))):
    item = private_ds[i]
    print(i, item["id"], item["path"], len(item["wav"]), item["canonical"])

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Private samples: 856
0 nang-giai_1618731750770 audio_data/private_test/nang-giai_1618731750770.wav 32694 n a-0 ŋz z aː-3 iz
1 nang-giai_1618731781800 audio_data/private_test/nang-giai_1618731781800.wav 26749 n a-0 ŋz z aː-3 iz
2 nang-giai_1618732605210 audio_data/private_test/nang-giai_1618732605210.wav 28235 n a-0 ŋz z aː-3 iz
3 nang-giai_1618732684413 audio_data/private_test/nang-giai_1618732684413.wav 28235 n a-0 ŋz z aː-3 iz
4 nang-giai_1618732695387 audio_data/private_test/nang-giai_1618732695387.wav 23777 n a-0 ŋz z aː-3 iz


In [ ]:
@torch.no_grad()
def predict_private(loader):
    model.eval()

    all_ids = []
    all_paths = []
    all_preds = []

    for batch in tqdm(loader, desc="Private Predict"):
        input_values = batch["input_values"].to(DEVICE)
        canonical = batch["canonical"].to(DEVICE)

        logits = model(input_values, canonical)
        log_probs = F.log_softmax(logits, dim=-1)

        for i in range(log_probs.shape[0]):
            pred = greedy_decode(log_probs[i])
            all_preds.append(pred)
            all_ids.append(batch["ids"][i])
            all_paths.append(batch["paths"][i])

    return all_ids, all_paths, all_preds


private_ids, private_paths, private_preds = predict_private(private_loader)

print("Num predictions:", len(private_preds))
for i in range(min(20, len(private_preds))):
    print(i, private_ids[i], private_paths[i], "=>", private_preds[i])

Private Predict:   0%|          | 0/856 [00:00<?, ?it/s]

Num predictions: 856
0 nang-giai_1618731750770 audio_data/private_test/nang-giai_1618731750770.wav => n a-0 ŋz z aː-1 iz
1 nang-giai_1618731781800 audio_data/private_test/nang-giai_1618731781800.wav => n a-0 ŋz z aː-2 iz
2 nang-giai_1618732605210 audio_data/private_test/nang-giai_1618732605210.wav => n a-0 ŋz z aː-3 iz
3 nang-giai_1618732684413 audio_data/private_test/nang-giai_1618732684413.wav => n a-0 ŋz z aː-4 iz
4 nang-giai_1618732695387 audio_data/private_test/nang-giai_1618732695387.wav => n a-0 ŋz z aː-3 iz
5 nang-giai_1618732709490 audio_data/private_test/nang-giai_1618732709490.wav => n a-0 ŋz z aː-0 iz
6 nhan-loai_1618732856294 audio_data/private_test/nhan-loai_1618732856294.wav => ɲ ə-0 nz l w aː-1 iz
7 nhan-loai_1618732872207 audio_data/private_test/nhan-loai_1618732872207.wav => ɲ ə-0 nz l aː-2 iz
8 nhan-loai_1618732882238 audio_data/private_test/nhan-loai_1618732882238.wav => ɲ ə-0 nz l aː-3 iz
9 nhan-loai_1618732889501 audio_data/private_test/nhan-loai_1618732889501.wav

In [ ]:
csv_path = f"{OUT_DIR}/results.csv"
zip_path = f"{OUT_DIR}/prediction.zip"

results_df = pd.DataFrame({
    "id": private_ids,
    "path": private_paths,
    "predict": private_preds
})

results_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
    lineterminator="\n"
)

print("Saved CSV:", csv_path)
print(results_df.head(20))
print(results_df.shape)
print(results_df.columns.tolist())

assert results_df.shape[0] == private_df.shape[0]
assert list(results_df.columns) == ["id", "path", "predict"]

with open(csv_path, "r", encoding="utf-8") as f:
    print("CSV first line:", repr(f.readline()))

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname="results.csv")

with zipfile.ZipFile(zip_path, "r") as zf:
    print("ZIP content:", zf.namelist())
    with zf.open("results.csv") as f:
        print("ZIP first line:", repr(f.readline().decode("utf-8")))

files.download(zip_path)

Saved CSV: /content/drive/MyDrive/MDD/Private_Result/results.csv
                         id  \
0   nang-giai_1618731750770   
1   nang-giai_1618731781800   
2   nang-giai_1618732605210   
3   nang-giai_1618732684413   
4   nang-giai_1618732695387   
5   nang-giai_1618732709490   
6   nhan-loai_1618732856294   
7   nhan-loai_1618732872207   
8   nhan-loai_1618732882238   
9   nhan-loai_1618732889501   
10  nhan-loai_1618732898716   
11  nhan-loai_1618732907026   
12   noi-trai_1618732921176   
13   noi-trai_1618732962207   
14   noi-trai_1618733000841   
15   noi-trai_1618733013767   
16   noi-trai_1618733039043   
17   noi-trai_1618738065718   
18   diem-dai_1618733094458   
19   nao-biet_1618733487266   

                                                 path                predict  
0   audio_data/private_test/nang-giai_161873175077...     n a-0 ŋz z aː-1 iz  
1   audio_data/private_test/nang-giai_161873178180...     n a-0 ŋz z aː-2 iz  
2   audio_data/private_test/nang-giai_16187326

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>